# Bias and Constrained Learning Homework

In this homework we'll extend the constrained learning framework we used for mitigating bias in class to handle more complex situations. Specifically, we'll look at the case where the output prediction is not binary. As usual with these homeworks, there are three different levels which build on each other, each one corresponding to an increasing grade:

- The basic version of this homework involves implementing code to measure fairness over multiclass classification then measuring the results when using training a regular, unfair classifier. This version is good for a C.
- The B version of the homework involves training a classifier with some fairness constraints.
- For an A, we'll look at slightly more complicated approach to fair training.

First, we'll generate a dataset for which the sensitive attribute is binary and the output is multiclass.

In [1]:
import numpy as np
import torch
from torch import nn, optim

In [2]:
output_classes = 5

def generate_data():

    dataset_size = 10000
    dimensions = 40

    rng = np.random.default_rng()
    A = np.concatenate((np.zeros(dataset_size // 2), np.ones(dataset_size // 2)))
    rng.shuffle(A)
    X = rng.normal(loc=A[:,np.newaxis], scale=1, size=(dataset_size, dimensions))
    random_linear = np.array([
        -2.28156561, 0.24582547, -2.48926942, -0.02934924, 5.21382855, -1.08613209,
        2.51051602, 1.00773587, -2.10409448, 1.94385103, 0.76013416, -2.94430782,
        0.3289264, -4.35145624, 1.61342623, -1.28433588, -2.07859612, -1.53812125,
        0.51412713, -1.34310334, 4.67174476, 1.67269946, -2.07805413, 3.46667731,
        2.61486654, 1.75418209, -0.06773796, 0.7213423, 2.43896438, 1.79306807,
        -0.74610264, 2.84046827,  1.28779878, 1.84490263, 1.6949681, 0.05814582,
        1.30510732, -0.92332861,  3.00192177, -1.76077192
    ])
    good_score = (X @ random_linear) ** 2 / 2
    qs = np.quantile(good_score, (np.array(range(1, output_classes))) / output_classes)
    Y = np.digitize(good_score, qs)

    return X, A, Y

X, A, Y = generate_data()

In [3]:
print("Total:", [(Y == k).sum() for k in range(output_classes)])
print("A=0:", [((Y == k) & (A == 0)).sum() for k in range(output_classes)])
print("A=1:", [((Y == k) & (A == 1)).sum() for k in range(output_classes)])

Total: [2000, 2000, 2000, 2000, 2000]
A=0: [1368, 1323, 1109, 811, 389]
A=1: [632, 677, 891, 1189, 1611]


This last cell shows the total number of data points in each output category (it should be 2000 each) as well as a breakdown of each output category for the $A=0$ group and the $A=1$ group. Note that the $A=1$ group is much more likely to be assigned to the categories with higher index.

## Fairness Definition (C)

Let's write some code to measure a few different forms of bias in our classifier. Demographic parity, which requires $P(R = r \mid A = 0) = P(R = r \mid A = 1)$ for all possible output classes $0 \le r < K$, and predictive parity which requires $P(Y=r \mid A = 0, R = r) = P(Y=r \mid A = 1, R = r)$. In the the functions below,

- `R` is a matrix where each row represents a probability distribution over the classes `0` to `K - 1`. That is, `R` is the output of our neural network _after_ a softmax layer.
- `A` is a vector of sensitive attributes. Each element is either `0` or `1`.
- `Y` is a vector of measured output classes, each element is between `0` and `K - 1`.

These functions should return an array of length `K` where each element of the array represents a measure of bias for _one_ of the output classes. For example, for demographic parity, the value in the output array at index `i` should be $P(R = i \mid A = 1) - P(R = i \mid A = 0)$.

Note that predictive parity is a bit different than the equalized odds measure I included in the solution to the bias lab. In particular, in the lab we used filtering to represent conditional probabilities, so $P(R=1 \mid A=0)$ was measured by `probs[A==0].mean()` for example. Now we can't do that directly since the predictive parity expression is conditioned on $R$ which is continuous. You'll need to instead use Bayes' rule and/or the definition of conditional probability to rearrange the predicitive parity equation until it's something we can measure. It's quite tricky to do this for all classes in one call, so it's okay to loop over the classes and compute the predictive parity for each on separately.

In [63]:
def demographic_parity(R, A):
    return R[A == 1].mean(axis=0) - R[A == 0].mean(axis=0)

def predictive_parity(R, A, Y):
    probs = []
    for i in range(output_classes): 
        # P(Y=i)
        probY = (Y==i).mean()
        # P(R=i, A=0 | Y=i)
        probRAY_0 = R[((A==0) & (Y==i)),i].mean() * (A==0).mean()
        # P(R=i, A=0)
        probRandA_0 = R[A==0,i].mean() * (A==0).mean()
        # P(R=i, A=1 | Y=i)
        probRAY_1 = R[((A==1) & (Y==i)),i].mean() * (A==1).mean()
        # P(R=i, A=1)
        probRandA_1 = R[A==1,i].mean() * (A==1).mean()

        prob_1 = (probRAY_1 * probY) / probRandA_1
        prob_0 = (probRAY_0 * probY) / probRandA_0
        predparity = prob_1 - prob_0
        probs.append(predparity)
    return probs # it outputs an array of tensors but the tensors say the calculated value which i think looks reasonable

Now we'll train a classifier on this dataset without any fairness constraints for comparison. This code is already complete.

In [64]:
class MLP(nn.Module):

    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(40, 256),
            nn.ReLU(),
            nn.Linear(256, 5)
        )

    def forward(self, x):
        return self.model(x)

In [65]:
def train_unfair(lr=1e-1, epochs=200):
    
    network = MLP()
    loss = nn.CrossEntropyLoss()
    opt = optim.SGD(network.parameters(), lr=lr)
    data_in = torch.tensor(X).float()
    data_out = torch.tensor(Y)
    
    for i in range(epochs):
        preds = network(data_in)
        loss_val = loss(preds, data_out)
        opt.zero_grad()
        loss_val.backward()
        opt.step()

        if (i+1) % 20 == 0:
            acc = (preds.argmax(dim=1) == data_out).float().mean()
            probs = nn.functional.softmax(preds, dim=1)
            print("Epoch:", i, "Accuracy:", acc.item(), "Bias:", demographic_parity(probs, A))

    return network

In [66]:
model = train_unfair(lr=5e-1, epochs=300)

Epoch: 19 Accuracy: 0.32850000262260437 Bias: tensor([-0.1686, -0.1478, -0.0778,  0.0201,  0.3741], grad_fn=<SubBackward0>)
Epoch: 39 Accuracy: 0.34689998626708984 Bias: tensor([-0.1884, -0.1623, -0.0836,  0.0432,  0.3911], grad_fn=<SubBackward0>)
Epoch: 59 Accuracy: 0.3833000063896179 Bias: tensor([-0.1999, -0.1700, -0.0838,  0.0559,  0.3978], grad_fn=<SubBackward0>)
Epoch: 79 Accuracy: 0.4300999939441681 Bias: tensor([-0.2028, -0.1730, -0.0833,  0.0536,  0.4055], grad_fn=<SubBackward0>)
Epoch: 99 Accuracy: 0.5051000118255615 Bias: tensor([-0.1955, -0.1691, -0.0810,  0.0390,  0.4066], grad_fn=<SubBackward0>)
Epoch: 119 Accuracy: 0.5810999870300293 Bias: tensor([-0.1889, -0.1648, -0.0756,  0.0388,  0.3905], grad_fn=<SubBackward0>)
Epoch: 139 Accuracy: 0.6310999989509583 Bias: tensor([-0.1869, -0.1624, -0.0734,  0.0410,  0.3817], grad_fn=<SubBackward0>)
Epoch: 159 Accuracy: 0.6628999710083008 Bias: tensor([-0.1860, -0.1597, -0.0717,  0.0428,  0.3746], grad_fn=<SubBackward0>)
Epoch: 179 

In [67]:
p = model(torch.tensor(X).float()).argmax(dim=1)
print("Total:", [(p == k).sum().item() for k in range(output_classes)])
print("A=0:", [((p == k) & (A == 0)).sum().item() for k in range(output_classes)])
print("A=1:", [((p == k) & (A == 1)).sum().item() for k in range(output_classes)])

Total: [1989, 1693, 2535, 668, 3115]
A=0: [1412, 1225, 1410, 297, 656]
A=1: [577, 468, 1125, 371, 2459]


This classifier is probably not going to be _extremely_ accurate, but you should be able to see the bias from the dataset reflected here. Let's also measure the bias using your two functions from above.

In [68]:
p = torch.nn.functional.softmax(model(torch.tensor(X).float()), dim=1)
print(demographic_parity(p, A))
print(predictive_parity(p, A, Y))

tensor([-0.1740, -0.1325, -0.0728,  0.0269,  0.3523], grad_fn=<SubBackward0>)
[tensor(0.6898, grad_fn=<SubBackward0>), tensor(0.2673, grad_fn=<SubBackward0>), tensor(0.0972, grad_fn=<SubBackward0>), tensor(-0.2982, grad_fn=<SubBackward0>), tensor(-1.0059, grad_fn=<SubBackward0>)]


## Fair Training (B)

Now we'll extend our fair training approach from the lab to the multiclass setting. Now since we have a bias measure for _each_ possible output class, we essentially have `output_classes` constraints that we need to satisfy. We can handle this within our Lagrange multiplier framework by simply adding extra multipliers for each constraint. That is, our new learning problem is

$$
\arg\min_\beta \max_\lambda \left ( L(\beta) + \sum_i \lambda_i g_i(\beta) \right )
$$

$$
= \arg\min_\beta \max_\lambda \left ( L(\beta) + \sum_i \lambda_i \left ( P_\beta [ R = i \mid A = 1 ] - P_\beta [ R = i \mid A = 0 ] \right ) \right )
$$

Our `demographic_parity` function gives us a vector representing $g_i(\beta)$, so now all we need to do is replace our single parameter $\lambda$ from the lab with a vector then compute the dot product of $\lambda$ with our demographic parity measure.

In [54]:
def train_fair(lr=1e-1, lam_lr=1, epochs=200):
    
    network = MLP()
    lam = nn.Parameter(torch.zeros(output_classes))
    loss = nn.CrossEntropyLoss()
    opt = optim.SGD(network.parameters(), lr=lr)
    lam_opt = optim.SGD([lam], lr=lam_lr, maximize=True)
    data_in = torch.tensor(X).float()
    data_out = torch.tensor(Y)
    
    for i in range(epochs):

        # Compute the loss value as defined in the Lagrangian above
        preds = network(data_in)
        loss_val = loss(preds, data_out)
        probs = nn.functional.softmax(preds, dim=1)
        bias = demographic_parity(probs, A)
        loss_val += torch.dot(lam, bias).sum() 
        # torch.dot should do the dot product of two 1d tensors, making it compute it element wise like in the equation above
        
        opt.zero_grad()
        lam_opt.zero_grad()
        loss_val.backward()
        opt.step()
        lam_opt.step()

        if (i+1) % 20 == 0:
            acc = (preds.argmax(dim=1) == data_out).float().mean()
            probs = nn.functional.softmax(preds, dim=1)
            print("Epoch:", i, "Accuracy:", acc.item(), "Bias:", demographic_parity(probs, A), "Lambda:", lam.max().item())

    return network

In [55]:
model = train_fair(lr=5e-1, lam_lr=3e-1, epochs=300)

Epoch: 19 Accuracy: 0.23810000717639923 Bias: tensor([-0.0303, -0.0290, -0.0097,  0.0038,  0.0652], grad_fn=<SubBackward0>) Lambda: 0.44183987379074097
Epoch: 39 Accuracy: 0.313400000333786 Bias: tensor([-0.0495, -0.0394, -0.0183,  0.0194,  0.0878], grad_fn=<SubBackward0>) Lambda: 0.4571970999240875
Epoch: 59 Accuracy: 0.38429999351501465 Bias: tensor([-0.0534, -0.0396, -0.0108,  0.0271,  0.0767], grad_fn=<SubBackward0>) Lambda: 0.4760802686214447
Epoch: 79 Accuracy: 0.4876999855041504 Bias: tensor([-0.0635, -0.0448, -0.0113,  0.0212,  0.0984], grad_fn=<SubBackward0>) Lambda: 0.5058990120887756
Epoch: 99 Accuracy: 0.5728999972343445 Bias: tensor([-0.0713, -0.0455, -0.0053,  0.0148,  0.1073], grad_fn=<SubBackward0>) Lambda: 0.5989150404930115
Epoch: 119 Accuracy: 0.6211000084877014 Bias: tensor([-0.0719, -0.0402,  0.0006,  0.0104,  0.1012], grad_fn=<SubBackward0>) Lambda: 0.6915413737297058
Epoch: 139 Accuracy: 0.6353999972343445 Bias: tensor([-0.0700, -0.0344,  0.0029,  0.0066,  0.0949

In [56]:
p = model(torch.tensor(X).float()).argmax(dim=1)
print("Total:", [(p == k).sum().item() for k in range(output_classes)])
print("A=0:", [((p == k) & (A == 0)).sum().item() for k in range(output_classes)])
print("A=1:", [((p == k) & (A == 1)).sum().item() for k in range(output_classes)])

Total: [2594, 2099, 628, 3741, 938]
A=0: [1208, 968, 307, 1972, 545]
A=1: [1386, 1131, 321, 1769, 393]


## Fair Training via KL-Divergence (A)

Let's look back at our definition of demographic parity for the multiclass setting: $P(R = r \mid A = 0) = P(R = r \mid A = 1)$ for all possible output classes $r$. we could also express this by asserting $P(\cdot \mid A = 0)$ and $P(\cdot \mid A = 1)$ should be identical probability distributions. A natural measure of bias then would be to compute the KL-divergence between these two distributions, since KL-divergence is a measure of how "different" two distributions are. That is, we'll now solve the problem

$$
\arg\min_\beta \max_\lambda \left ( L(\beta) + \lambda D_{\textrm{KL}} \left( P(\cdot \mid A = 0) \ \| \ P(\cdot \mid A = 1) \right) \right )
$$

However, this introduces a new complication. The KL-divergence is never negative and can only be zero if the two distributions are identical (we proved this in our first homework of the semester). That means there's no way for $\lambda$ to ever decrease, and it will just go up forever. We can solve this by allowing a small deviation in our constrained optimization problem:

$$
\begin{align}
\arg\min_\beta &\ L(\beta) \\
\text{s.t.} &\ D_{\textrm{KL}} \left( P(\cdot \mid A = 0) \ \| \ P(\cdot \mid A = 1) \right) \le \epsilon
\end{align}
$$

We can still represent this using a Lagrange multiplier:

$$
\arg\min_\beta \max_{\lambda \ge 0} \left ( L(\beta) + \lambda \left ( D_{\textrm{KL}} \left( P(\cdot \mid A = 0) \ \| \ P(\cdot \mid A = 1) \right) - \epsilon \right ) \right )
$$

Your task now is to represent this optimization problem in the code below. I've taken care of clipping $\lambda$ to zero for you since it's not something we've looked at in class.

In [13]:
def train_kl(lr=1e-1, lam_lr=1, epochs=300, epsilon=0.1):
    
    network = MLP()
    lam = nn.Parameter(torch.tensor(0.0))
    loss = nn.CrossEntropyLoss()
    opt = optim.SGD(network.parameters(), lr=lr)
    lam_opt = optim.SGD([lam], lr=lam_lr, maximize=True)
    data_in = torch.tensor(X).float()
    data_out = torch.tensor(Y)
    
    for i in range(epochs):

        # Implement the loss function above here.
        loss_val = ???
        
        opt.zero_grad()
        lam_opt.zero_grad()
        loss_val.backward()
        opt.step()
        lam_opt.step()
        with torch.no_grad():
            lam.clamp_(min=0)

        if (i+1) % 20 == 0:
            acc = (preds.argmax(dim=1) == data_out).float().mean()
            print("Epoch:", i, "Accuracy:", acc.item(), "Divergence:", kl_div.item(), "Lambda:", lam.item())

    return network

SyntaxError: invalid syntax (2335916144.py, line 14)

In [ ]:
model = train_kl(lr=3e-1, lam_lr=1, epsilon=0.02)

In [ ]:
p = model(torch.tensor(X).float()).argmax(dim=1)
print("Total:", [(p == k).sum().item() for k in range(output_classes)])
print("A=0:", [((p == k) & (A == 0)).sum().item() for k in range(output_classes)])
print("A=1:", [((p == k) & (A == 1)).sum().item() for k in range(output_classes)])